# Baseline retrieval vs cross-encoder re-ranking

Document: FIA 2026 Formula 1 Regulations, Section B (Sporting), Issue 09, dated 1 October 2026.

Both pipelines run over the same chunked text and use the same prompt and the same language model.

1. Pipeline 1: embed the question with a bi-encoder, take the 10 nearest chunks by cosine
   similarity, send the top 3 to the model (Answer 1).
2. Pipeline 2: score the same 10 chunks with a cross-encoder, sort by that score, send the new
   top 3 to the model (Answer 2).

Changes made to the source before indexing (also listed in the README):

* Pages 1 to 3 (cover, colour legend, contents) are not indexed.
* Pages 95 to 101 (approved changes for 2027 to 2029, not in force) are not indexed.
* Struck-out text is removed. The PDF marks deleted rules with a magenta line through the text,
  and plain text extraction would index them as if they were still in force.

To run: install `requirements.txt`, then run all cells. The answer step needs `OPENCODE_API_KEY`
in the environment. Without it the notebook uses the saved answers in `results/answers_cache.json`.

## 1. Setup

In [1]:
import hashlib, json, os, re, statistics, sys, time, urllib.request, uuid
from datetime import date
from pathlib import Path

if "google.colab" in sys.modules:
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pymupdf==1.28.2", "sentence-transformers==6.1.0", "faiss-cpu==1.15.1"], check=True)

# torch and faiss each bring their own OpenMP runtime. Import torch first: the other order
# crashes on Apple Silicon.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
import numpy as np
import torch
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import pymupdf

tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1018 23:07:55.265000 56629 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


In [2]:
PDF_URL = ("https://www.fia.com/system/files/documents/"
           "fia_2026_f1_regulations_-_section_b_sporting_-_iss_09_-_2026-10-01.pdf")
PDF_SHA256 = "5e156f50f1ca529d43551f52e82f5f2872bdfbe088e96fab39b861f6b0704b85"
PDF_PATH = Path("data/fia_2026_f1_section_b_sporting_iss09.pdf")
FIRST_PAGE, LAST_PAGE = 4, 94          # 1-based, inclusive

BI_ENCODER = "BAAI/bge-small-en-v1.5"
CROSS_ENCODER = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# bge asks for this phrase in front of short queries. Passages get no phrase, and the
# cross-encoder reads the plain question.
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "
TOP_K, TOP_N = 10, 3
MAX_CHUNK_TOKENS = 400                 # leaves room for the question in the cross-encoder's 512

GEN_MODEL = "claude-haiku-5-5"
GEN_MAX_TOKENS = 800                   # output budget; the model's own thinking tokens count against it
GEN_URL = "https://opencode.ai/inference/go/anthropic/v1/messages"
CACHE_PATH = Path("results/answers_cache.json")
CACHE_URL = ("https://raw.githubusercontent.com/abhinavborah/rag-rerank-eval/main/"
             "results/answers_cache.json")      # used when the notebook runs without the repository

## 2. Source document

In [3]:
def sha256_of(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

PDF_PATH.parent.mkdir(exist_ok=True)
if not PDF_PATH.exists():
    req = urllib.request.Request(PDF_URL, headers={"User-Agent": "Mozilla/5.0"})
    PDF_PATH.write_bytes(urllib.request.urlopen(req, timeout=120).read())
assert sha256_of(PDF_PATH) == PDF_SHA256, "the downloaded PDF is not the pinned edition"
doc = pymupdf.open(PDF_PATH)
print(doc.metadata["title"], "|", len(doc), "pages")

FIA 2026 F1 Regulations - Section B [Sporting] | 101 pages


## 3. Text extraction

Each page is read character by character. A character is dropped when a magenta bar crosses the
middle of it, which is how the PDF shows deleted text. Headers and footers are skipped. What is
left is a list of visual lines with their position and font, which the next step groups into
clauses.

In [4]:
MAGENTA = (1.0, 0.0, 1.0)
BLUE = 0x2D5F                          # colour of clause labels and headings
LABEL_COLORS = (BLUE, 0xFF00FF)        # a clause added in this issue has a magenta label

def strike_bars(page):
    bars = []
    for d in page.get_drawings():
        r, col = d["rect"], d.get("color") or d.get("fill")
        if r.height < 1.6 and r.width > 3 and col and tuple(round(c, 2) for c in col) == MAGENTA:
            bars.append(r)
    return bars

def is_struck(bbox, bars):
    x0, y0, x1, y1 = bbox
    cx = (x0 + x1) / 2
    lo, hi = y0 + (y1 - y0) * 0.25, y0 + (y1 - y0) * 0.75
    return any(lo <= (r.y0 + r.y1) / 2 <= hi and r.x0 - 1 <= cx <= r.x1 + 1 for r in bars)

def page_lines(page):
    """Visible lines of one page: dicts with y, x, size, bold, color and text."""
    bars = strike_bars(page)
    out, dropped = [], 0
    for block in page.get_text("rawdict")["blocks"]:
        for line in block.get("lines", []):
            y0 = line["bbox"][1]
            if y0 < 50 or y0 > page.rect.height - 65:       # running header and footer
                continue
            chars, first = [], None
            for span in line["spans"]:
                for ch in span["chars"]:
                    if is_struck(ch["bbox"], bars):
                        dropped += 1
                        continue
                    chars.append(ch["c"])
                    if first is None and ch["c"].strip():
                        first = (span, ch["bbox"][0])
            text = re.sub(r"\s+", " ", "".join(chars)).strip()
            if not text or first is None:
                continue
            span, x = first
            if x < 85 and re.fullmatch(r"\d+(\.\d+)*", text):   # row numbers in the appendix tables
                continue
            out.append({"y": y0, "x": x, "size": span["size"], "bold": "Bold" in span["font"],
                        "color": span["color"], "text": text})
    # A 9 pt clause label and the 11 pt first line next to it have tops about 2 pt apart, so sorting on
    # the raw top can put the body line ahead of its own label. Lines whose tops are within 3 pt
    # are one visual row, ordered left to right.
    out.sort(key=lambda l: l["y"])
    rows, row = [], []
    for l in out:
        if row and l["y"] - row[0]["y"] > 3:
            rows.append(row)
            row = []
        row.append(l)
    if row:
        rows.append(row)
    return [l for r in rows for l in sorted(r, key=lambda l: l["x"])], dropped

lines, struck_chars = [], 0
for pno in range(FIRST_PAGE - 1, LAST_PAGE):
    pl, n = page_lines(doc[pno])
    struck_chars += n
    for l in pl:
        l["page"] = pno + 1
    lines += pl
print(len(lines), "lines kept,", struck_chars, "struck characters removed")

full = " ".join(l["text"] for l in lines)
for marker in ("SECTION B: SPORTING REGULATIONS", "Fédération Internationale", "Issue 09",
               "2026 Formula 1 Regulations - Section B", "01 October 2026"):
    assert marker not in full, marker

4253 lines kept, 3135 struck characters removed


## 4. Chunking

The regulations are numbered (B1.9.3 and so on). A chunk is one clause, with its article,
section and clause number written in front, so a chunk that says "the penalty" still carries its
heading. A clause longer than the token limit is split at paragraph boundaries, with the last
paragraph repeated at the start of the next piece. Paragraphs that are still too long are split
at sentence ends.

In [5]:
tok = SentenceTransformer(BI_ENCODER, device="cpu").tokenizer     # token counts use the model's own tokenizer
tok.model_max_length = 10**6                                  # only counting tokens here, nothing is truncated

def n_tokens(text):
    return len(tok(text, add_special_tokens=True)["input_ids"])

def label_kind(l):
    t = l["text"]
    if l["bold"] and l["color"] in LABEL_COLORS and l["x"] < 60:
        if re.fullmatch(r"B\d+(\.\d+){2,}", t):
            return "clause"
        if re.fullmatch(r"B\d+\.\d+", t):
            return "section"
    if re.match(r"(ARTICLE|APPENDIX) B\d+:", t) and l["x"] < 60:
        return "article"
    return None

blocks = []          # one per clause (or per run of text under a heading)
state = {"article": None, "section": None, "sec_title": None, "clause": None}

def new_block():
    blocks.append({**state, "lines": []})

new_block()
last_section_y = None
for l in lines:
    kind = label_kind(l)
    if kind == "article":
        state.update(article=l["text"], section=None, sec_title=None, clause=None)
        new_block()
    elif kind == "section":
        state.update(section=l["text"], sec_title="", clause=None)
        last_section_y = (l["page"], round(l["y"]))
        new_block()
    elif kind == "clause":
        state.update(clause=l["text"])
        new_block()
    elif (last_section_y == (l["page"], round(l["y"])) and l["bold"] and l["color"] in LABEL_COLORS):
        state["sec_title"] = l["text"]
        blocks[-1]["sec_title"] = l["text"]
    else:
        blocks[-1]["lines"].append(l)

# Every line that looks like a clause or section number in the margin must have been read as a label,
# and every section must have a title.
missed = [l for l in lines if l["x"] < 60 and re.fullmatch(r"B\d+(\.\d+)+", l["text"]) and not label_kind(l)]
assert not missed, missed[:3]
assert all(b["sec_title"] for b in blocks if b["section"]), "a section lost its title"

# Boundary check: a clause should not start in the middle of a sentence.
starts_lower = [b["clause"] for b in blocks if b["clause"] and b["lines"] and b["lines"][0]["text"][:1].islower()
                and not re.match(r"[a-z]{1,3}\.", b["lines"][0]["text"])]
print("clauses starting lower case:", starts_lower)

SUBITEM = re.compile(r"^([a-z]|[ivx]+|\d+)\.\s")

def paragraphs(block_lines):
    paras, prev = [], None
    for l in block_lines:
        starts_new = (prev is None or SUBITEM.match(l["text"])
                      or (l["page"] == prev["page"] and l["y"] - prev["y"] > 18))
        if starts_new:
            paras.append([l["page"], [l["text"]]])
        else:
            paras[-1][1].append(l["text"])
        prev = l
    return [(" ".join(t), pg) for pg, t in paras]

def crumb(b):
    parts = [b["article"]]
    if b["section"]:
        parts.append(f"{b['section']} {b['sec_title'] or ''}".strip())
    if b["clause"]:
        parts.append(b["clause"])
    return " > ".join(p for p in parts if p)

def split_long(para, budget):
    sentences, pieces, cur = re.split(r"(?<=[.;:])\s+", para), [], ""
    for s in sentences:
        cand = f"{cur} {s}".strip()
        if cur and n_tokens(cand) > budget:
            pieces.append(cur)
            cur = s
        else:
            cur = cand
    return pieces + [cur]

# A heading block that holds only a short note (for example the committee line under an article
# title) is merged into the block after it, so it does not become a chunk of its own.
for i, b in enumerate(blocks[:-1]):
    if b["clause"] is None and b["lines"] and             n_tokens(" ".join(l["text"] for l in b["lines"])) < 45:
        blocks[i + 1]["lines"] = b["lines"] + blocks[i + 1]["lines"]
        b["lines"] = []

chunks, seen = [], {}
for b in blocks:
    paras = paragraphs(b["lines"])
    if not paras:
        continue
    head = crumb(b)
    title = paras[0][0] if (len(paras[0][0].split()) <= 12 and not paras[0][0].endswith(".")
                            and not paras[0][0].startswith("Advisory Committee")) else ""
    budget = MAX_CHUNK_TOKENS - n_tokens(head + " > " + title) - 2
    units = []                                   # (text, page) pairs
    for p, pg in paras:
        units += [(x, pg) for x in split_long(p, budget)] if n_tokens(p) > budget else [(p, pg)]
    groups, cur = [], []
    for u in units:
        if cur and n_tokens(" ".join(t for t, _ in cur + [u])) > budget:
            groups.append(cur)
            cur = [cur[-1], u] if n_tokens(cur[-1][0] + " " + u[0]) <= budget else [u]
        else:
            cur.append(u)
    groups.append(cur)
    base = b["clause"] or (b["section"] or (b["article"] or "front")).split(":")[0]
    seen[base] = seen.get(base, 0) + 1
    for k, g in enumerate(groups, 1):
        cid = base if len(groups) == 1 else f"{base}#{k}"
        if seen[base] > 1:
            cid += f"@{seen[base]}"
        part_head = head + " > " + title if (k > 1 and title) else head   # later pieces keep the clause title
        text = part_head + "\n" + "\n".join(t for t, _ in g)
        pages = [pg for _, pg in g]
        chunks.append({"id": cid, "text": text, "tokens": n_tokens(text), "pages": (min(pages), max(pages))})

sizes = [c["tokens"] for c in chunks]
print(len(chunks), "chunks | tokens min/median/max:", min(sizes), int(statistics.median(sizes)), max(sizes))
assert max(sizes) <= MAX_CHUNK_TOKENS + 10
assert len({c["id"] for c in chunks}) == len(chunks)

# The saved answers depend on the exact chunk text. If extraction or chunking changes, this fails.
corpus_hash = hashlib.sha256("\n".join(f"{c['id']}\t{c['text']}" for c in chunks).encode()).hexdigest()
EXPECTED_CHUNKS, EXPECTED_CORPUS_SHA256 = 375, "ac164cf33589cd0daf4f91928023a02f74af9bb3fb4881fc5107ad8c45ea407e"
print("corpus hash", corpus_hash[:16])
if EXPECTED_CHUNKS is not None:
    assert (len(chunks), corpus_hash) == (EXPECTED_CHUNKS, EXPECTED_CORPUS_SHA256), "the chunked corpus changed"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9095.50it/s]

clauses starting lower case: []


375 chunks | tokens min/median/max: 32 138 395
corpus hash ac164cf33589cd0d


## 5. Index and the two pipelines

In [6]:
bi = SentenceTransformer(BI_ENCODER, device="cpu")
cross = CrossEncoder(CROSS_ENCODER, device="cpu", max_length=512)
assert max(c["tokens"] for c in chunks) <= bi.max_seq_length

emb = bi.encode([c["text"] for c in chunks], batch_size=32, normalize_embeddings=True,
                show_progress_bar=False).astype("float32")
index = faiss.IndexFlatIP(emb.shape[1])        # exact search; inner product of unit vectors = cosine
index.add(emb)
print(index.ntotal, "vectors of dimension", emb.shape[1])

# The cross-encoder cuts its input at 512 tokens. Check the longest chunk with a long question.
probe = " ".join(["word"] * 40)
longest = max(len(cross.tokenizer(probe, c["text"])["input_ids"]) for c in chunks)
assert longest <= 512, longest
print("longest question + chunk pair:", longest, "tokens")

# FAISS must agree with a plain matrix product, since the search is exact.
for q in ("pit lane speed limit", "tyre allocation for the race"):
    qv = bi.encode([QUERY_PREFIX + q], normalize_embeddings=True).astype("float32")
    assert list(index.search(qv, TOP_K)[1][0]) == list(np.argsort(-(emb @ qv[0]))[:TOP_K])

def retrieve(query, k=TOP_K):
    """Pipeline 1, first stage: top-k chunks by cosine similarity."""
    t0 = time.perf_counter()
    q = bi.encode([QUERY_PREFIX + query], normalize_embeddings=True).astype("float32")
    t1 = time.perf_counter()
    scores, ids = index.search(q, k)
    t2 = time.perf_counter()
    return list(ids[0]), list(scores[0]), {"embed_s": t1 - t0, "search_s": t2 - t1}

def rerank(query, ids):
    """Pipeline 2, second stage: the same chunks, ordered by the cross-encoder."""
    t0 = time.perf_counter()
    s = cross.predict([(query, chunks[i]["text"]) for i in ids], batch_size=len(ids),
                      show_progress_bar=False)
    t1 = time.perf_counter()
    order = np.argsort(-s)
    return [ids[o] for o in order], [float(s[o]) for o in order], {"rerank_s": t1 - t0}

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 21281.11it/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 9937.54it/s]

375 vectors of dimension 384
longest question + chunk pair: 436 tokens


## 6. Answer generation

Both pipelines use this prompt and this model. Passages are numbered, the model may only use
them, and it is told to say so when they do not answer the question. Answers are saved in
`results/answers_cache.json`, keyed by model and prompt text, so a run without the API key
reproduces the same answers.

In [7]:
SYSTEM = ("Answer the question using only the numbered passages. Cite the passages you use, "
          "like [1]. If the passages do not answer the question, say that the document "
          "does not cover it.")

def build_prompt(query, ids):
    ctx = "\n\n".join(f"[{n}] {chunks[i]['text']}" for n, i in enumerate(ids, 1))
    return f"Context:\n{ctx}\n\nQuestion: {query}"

def load_cache(path=CACHE_PATH):
    if Path(path).exists():
        return json.loads(Path(path).read_text())
    if path != CACHE_PATH:
        return {}
    try:
        return json.loads(urllib.request.urlopen(CACHE_URL, timeout=30).read())
    except Exception:
        return {}

def cache_key(user, max_tokens=GEN_MAX_TOKENS):
    # The default budget keeps the original key; any other budget gets its own key.
    extra = "" if max_tokens == GEN_MAX_TOKENS else f"{max_tokens}\n"
    return hashlib.sha256(f"{GEN_MODEL}\n{extra}{SYSTEM}\n{user}".encode()).hexdigest()

def generate(user, max_tokens=GEN_MAX_TOKENS, cache_path=CACHE_PATH):
    key = cache_key(user, max_tokens)
    cache = load_cache(cache_path)
    if key in cache:
        return cache[key]["answer"], cache[key]["gen_s"], True
    api_key = os.environ.get("OPENCODE_API_KEY")
    if not api_key:
        raise RuntimeError("no saved answer for this prompt and OPENCODE_API_KEY is not set")
    body = json.dumps({"model": GEN_MODEL, "max_tokens": max_tokens, "system": SYSTEM,
                       "messages": [{"role": "user", "content": user}]}).encode()
    req = urllib.request.Request(GEN_URL, data=body, method="POST", headers={
        "content-type": "application/json", "x-api-key": api_key, "anthropic-version": "2023-06-01",
        "user-agent": "rag-rerank-eval/1.0", "x-opencode-session": SESSION})
    t0 = time.perf_counter()
    reply = json.loads(urllib.request.urlopen(req, timeout=120).read())
    gen_s = time.perf_counter() - t0
    answer = "".join(b.get("text", "") for b in reply["content"] if b["type"] == "text").strip()
    cache[key] = {"model": GEN_MODEL, "date": date.today().isoformat(), "answer": answer,
                  "gen_s": gen_s, "usage": reply.get("usage")}
    Path(cache_path).parent.mkdir(exist_ok=True)
    Path(cache_path).write_text(json.dumps(cache, indent=1))
    return answer, gen_s, False

SESSION = str(uuid.uuid4())

## 7. Running a query through both pipelines

In [8]:
def run_query(query, with_answers=True):
    ids, sims, t_ret = retrieve(query)
    new_ids, ce_scores, t_rr = rerank(query, ids)
    res = {"query": query, "baseline_ids": ids[:TOP_N], "rerank_ids": new_ids[:TOP_N],
           "rank_shift": {chunks[i]["id"]: (ids.index(i) + 1, new_ids.index(i) + 1) for i in ids},
           "cosine": dict(zip((chunks[i]["id"] for i in ids), sims)),
           "cross_score": dict(zip((chunks[i]["id"] for i in new_ids), ce_scores)),
           "time": {**t_ret, **t_rr}}
    if with_answers:
        res["answer_1"], res["gen_s_1"], _ = generate(build_prompt(query, res["baseline_ids"]))
        res["answer_2"], res["gen_s_2"], _ = generate(build_prompt(query, res["rerank_ids"]))
    return res

def show(res):
    print("QUERY:", res["query"])
    print(f"{'chunk':<14}{'baseline':>9}{'rerank':>8}{'cosine':>9}{'cross':>9}")
    for cid, (a, b) in sorted(res["rank_shift"].items(), key=lambda kv: kv[1][1]):
        print(f"{cid:<14}{a:>9}{b:>8}{res['cosine'][cid]:>9.3f}{res['cross_score'][cid]:>9.2f}")
    for k in ("answer_1", "answer_2"):
        if k in res:
            print(f"\n{k.upper().replace('_', ' ')}:\n{res[k]}")

## 8. Latency profile

Each stage is timed separately, after one warm-up call, with the models already loaded. The
numbers depend on the machine, so the hardware is printed with them.

In [9]:
import platform

def profile(queries, repeats=5):
    retrieve(queries[0]); rerank(queries[0], retrieve(queries[0])[0])      # warm-up
    rows = {"embed_s": [], "search_s": [], "rerank_s": []}
    for q in queries:
        for _ in range(repeats):
            ids, _, t1 = retrieve(q)
            _, _, t2 = rerank(q, ids)
            for k, v in {**t1, **t2}.items():
                rows[k].append(v)
    ms = {k: statistics.median(v) * 1000 for k, v in rows.items()}
    print(f"{platform.machine()} | {platform.processor() or platform.platform()} | torch {torch.__version__}")
    print(f"{len(queries)} queries x {repeats} runs, median per stage")
    print(f"  baseline retrieval (embed question + search): {ms['embed_s'] + ms['search_s']:8.1f} ms")
    print(f"  re-ranking ({TOP_K} pairs):                    {ms['rerank_s']:8.1f} ms")
    print(f"  pipeline 2 total (retrieval + re-ranking):    {ms['embed_s'] + ms['search_s'] + ms['rerank_s']:8.1f} ms")
    return ms

## 9. Queries and scoring

The query set is in `queries.json`: each question has a type, and a list of gold chunks. Gold is the
smallest set of chunks that together give a complete answer. Supporting material that is not needed
for a complete answer is noted in the file but not scored. The set was written and saved before any
of it was run.

Retrieval is scored on the ten retrieved chunks, for both pipelines:

* hit@3: at least one gold chunk is in the top 3.
* recall@3: gold chunks in the top 3, divided by the smaller of 3 and the number of gold chunks, so a
  query with four gold chunks can still reach 1.
* MRR: one over the rank of the best-placed gold chunk, 0 if none is in the top 10.
* recall@10 is not reported. Re-ranking only reorders the same ten chunks, so it is identical for
  both pipelines.

Queries with no gold chunk (out of scope) are not scored. They are only used to see whether the
answer refuses. Where both pipelines pass the same three chunks in a different order, any difference
between Answer 1 and Answer 2 is partly the language model's own variation, and such queries are
flagged.

In [10]:
queries = json.loads(Path("queries.json").read_text())
assert all(g in {c["id"] for c in chunks} for q in queries for g in q["gold"]), "unknown gold chunk"
LOG_PATH = Path("results/query_log.jsonl")

def score(q, res):
    out = {}
    for k, name in enumerate(("baseline", "rerank")):
        ranks = {g: res["rank_shift"].get(g, (None, None))[k] for g in q["gold"]}
        found = [r for r in ranks.values() if r is not None]
        out[name] = {"ranks": ranks,
                     "hit@3": any(r <= TOP_N for r in found),
                     "recall@3": sum(r <= TOP_N for r in found) / min(TOP_N, len(q["gold"])),
                     "rr": 1 / min(found) if found else 0.0}
    return out

def load_rows(path):
    return [json.loads(line) for line in Path(path).read_text().splitlines()]

def run_all(queries, log_path=LOG_PATH, with_answers=True):
    """Run every query through both pipelines and write one JSON line per query.
    A log is a record of one run, so an existing file is never overwritten."""
    if Path(log_path).exists():
        raise FileExistsError(f"{log_path} already exists; load it with load_rows or pass a new log_path")
    rows = []
    for q in queries:
        res = run_query(q["query"], with_answers=with_answers)
        row = {"id": q["id"], "type": q["type"], "query": q["query"], "gold": q["gold"],
               "baseline_top3": [chunks[i]["id"] for i in res["baseline_ids"]],
               "rerank_top3": [chunks[i]["id"] for i in res["rerank_ids"]],
               "rank_shift": {k: [int(a), int(b)] for k, (a, b) in res["rank_shift"].items()},
               "time_ms": {k: round(float(v) * 1000, 2) for k, v in res["time"].items()}}
        row["same_top3_set"] = set(row["baseline_top3"]) == set(row["rerank_top3"])
        row["same_top3_order"] = row["baseline_top3"] == row["rerank_top3"]
        for k in ("answer_1", "answer_2"):
            if k in res:
                row[k] = res[k]
        for k in ("gen_s_1", "gen_s_2"):
            if k in res:
                row[k] = round(float(res[k]), 2)
        if q["gold"]:
            row["scores"] = score(q, res)
        rows.append(row)
    log_path.parent.mkdir(exist_ok=True)
    log_path.write_text("".join(json.dumps(r) + "\n" for r in rows))
    return rows

def summarise(rows):
    groups = {}
    for r in rows:
        if "scores" in r:
            groups.setdefault(r["type"], []).append(r["scores"])
            groups.setdefault("ALL", []).append(r["scores"])
    mean = lambda L, p, key: statistics.mean(float(s[p][key]) for s in L)
    print(f"{'type':<13}{'n':>3} | hit@3 base  rerank | recall@3 base  rerank | MRR base  rerank")
    for t, L in groups.items():
        print(f"{t:<13}{len(L):>3} | {mean(L,'baseline','hit@3'):>11.2f}{mean(L,'rerank','hit@3'):>8.2f} | "
              f"{mean(L,'baseline','recall@3'):>14.2f}{mean(L,'rerank','recall@3'):>8.2f} | "
              f"{mean(L,'baseline','rr'):>8.2f}{mean(L,'rerank','rr'):>8.2f}")
    scored = [r for r in rows if "scores" in r]
    best = lambda r, p: r["scores"][p]["rr"]
    print("gold chunk outside the top 10 in both pipelines (re-ranking cannot fix):",
          [r["id"] for r in scored if best(r, "baseline") == 0])
    print("re-ranking moved the best gold chunk up:  ", [r["id"] for r in scored if best(r, "rerank") > best(r, "baseline")])
    print("re-ranking moved the best gold chunk down:", [r["id"] for r in scored if best(r, "rerank") < best(r, "baseline")])
    print("same top 3 in a different order (answer differences are partly noise):",
          [r["id"] for r in rows if r["same_top3_set"] and not r["same_top3_order"]])

## 10. Run all queries

Every query goes through both pipelines. The rows are written to `results/query_log.jsonl`, one per
query, with both rank lists, the top 3 of each pipeline, the timings and both answers. The file is the
record of the run reported in the README and is never overwritten: if it exists, the cell reads it
instead of running the queries again. The timings in it are single cold measurements; the latency
section below is measured separately on warm models.

In [11]:
rows = load_rows(LOG_PATH) if LOG_PATH.exists() else run_all(queries)
summarise(rows)

type           n | hit@3 base  rerank | recall@3 base  rerank | MRR base  rerank
control        3 |        1.00    1.00 |           1.00    1.00 |     1.00    1.00
ALL           29 |        0.86    0.83 |           0.75    0.71 |     0.67    0.74
keyword_trap   6 |        0.67    0.50 |           0.67    0.50 |     0.47    0.46
exception      9 |        0.89    0.89 |           0.70    0.67 |     0.70    0.83
negation       7 |        0.86    0.86 |           0.79    0.79 |     0.62    0.62
scattered      4 |        1.00    1.00 |           0.75    0.75 |     0.75    1.00
gold chunk outside the top 10 in both pipelines (re-ranking cannot fix): ['Q04', 'Q12', 'Q23']
re-ranking moved the best gold chunk up:   ['Q07', 'Q08', 'Q14', 'Q16', 'Q18', 'Q22', 'Q26', 'Q28']
re-ranking moved the best gold chunk down: ['Q05', 'Q19', 'Q20']
same top 3 in a different order (answer differences are partly noise): ['Q16', 'Q18']


In [12]:
def rank_table(rows):
    """Rank of each gold chunk among the ten retrieved, baseline to re-ranked. A dash means not retrieved."""
    for r in rows:
        if "scores" not in r:
            continue
        parts = []
        for g, rk in r["scores"]["baseline"]["ranks"].items():
            new = r["scores"]["rerank"]["ranks"][g]
            parts.append(f"{g} {rk or '-'}->{new or '-'}")
        print(f"{r['id']:<5}{r['type']:<13}{'; '.join(parts)}")

rank_table(rows)

Q01  control      B1.6.3#1 1->1
Q02  control      B2.5.2 1->1
Q03  control      B1.5.8 1->1
Q04  keyword_trap B5.5.4 -->-
Q05  keyword_trap B5.11.1 3->9
Q06  keyword_trap B1.8.6 1->1
Q07  keyword_trap B5.14.2#2 8->7
Q08  keyword_trap B6.3.8#1 3->2
Q09  keyword_trap B6.2.4 1->1
Q10  exception    B1.9.6#1 1->1
Q11  exception    B1.6.1#1 3->3; B5.3.3 1->1
Q12  exception    B5.1.6 -->-; B1.5.4 -->-
Q13  exception    B2.5.2 1->1; B5.10.7 -->-; B5.8.3 -->-; B5.9.5 -->-
Q14  exception    B5.13.2#2 -->-; B5.13.2#3 2->1; B5.13.4#1 3->6
Q15  exception    B3.5.3 1->1; B3.5.4 10->7; B3.5.5 8->9
Q16  exception    B1.5.10 2->1
Q17  exception    B3.2.3 1->1
Q18  exception    B1.7.2 3->2
Q19  negation     B1.5.9 2->3
Q20  negation     B1.6.1#1 1->2
Q21  negation     B5.1.5 1->1
Q22  negation     B5.7.1 3->1
Q23  negation     B3.3.3 -->-
Q24  negation     B1.6.2#3 1->1
Q25  negation     B1.9.7 2->2; B1.9.5#1 4->4
Q26  scattered    B1.9.5#1 2->1; B1.9.5#2 9->9
Q27  scattered    B1.9.6#4 1->1
Q28  scatte

## 11. Latency

In [13]:
latency = profile([q["query"] for q in queries], repeats=5)
gen = [r[k] for r in rows for k in ("gen_s_1", "gen_s_2") if k in r]
print(f"answer generation, median of {len(gen)} calls: {statistics.median(gen):.2f} s "
      f"(time of the original call, also kept when an answer is read from the saved file)")

arm64 | arm | torch 2.14.1
30 queries x 5 runs, median per stage
  baseline retrieval (embed question + search):      6.9 ms
  re-ranking (10 pairs):                        69.9 ms
  pipeline 2 total (retrieval + re-ranking):        76.8 ms
answer generation, median of 60 calls: 3.25 s (time of the original call, also kept when an answer is read from the saved file)


## 12. Score log of the frozen run

The per-query log holds ranks but not the scores behind them. This section reruns retrieval and
re-ranking only (no model calls), checks that every rank equals the logged rank, and saves the cosine
and cross-encoder scores to `results/score_log.jsonl`. If that file already exists the scores are
recomputed and compared with it, so the cell doubles as a reproducibility check. The two score scales
are not comparable with each other: cosine is in [-1, 1], the cross-encoder score is an unbounded logit.

In [14]:
SCORE_LOG = Path("results/score_log.jsonl")

def score_trace(query, k=TOP_K):
    """Retrieval and re-ranking of one query with their scores, no generation."""
    ids, sims, _ = retrieve(query, k)
    new_ids, ce, _ = rerank(query, ids)
    return {"baseline": [[chunks[i]["id"], round(float(x), 5)] for i, x in zip(ids, sims)],
            "reranked": [[chunks[i]["id"], round(float(x), 4)] for i, x in zip(new_ids, ce)]}

def save_or_verify(path, new_rows, tol=1e-3):
    """Write a log that does not exist yet; if it exists, check the new rows against it instead."""
    if not Path(path).exists():
        Path(path).parent.mkdir(exist_ok=True)
        Path(path).write_text("".join(json.dumps(r) + "\n" for r in new_rows))
        return "written"
    old = load_rows(path)
    assert [r["id"] for r in old] == [r["id"] for r in new_rows]
    for a, b in zip(old, new_rows):
        for side in ("baseline", "reranked"):
            assert [c for c, _ in a[side]] == [c for c, _ in b[side]], (a["id"], side)
            assert all(abs(x - y) <= tol for (_, x), (_, y) in zip(a[side], b[side])), (a["id"], side)
    return "verified"

score_rows = []
for r in rows:
    t = score_trace(r["query"])
    # The ranks must equal the ranks in the frozen log.
    for pos, side in ((0, "baseline"), (1, "reranked")):
        got = {c: n for n, (c, _) in enumerate(t[side], 1)}
        want = {c: v[pos] for c, v in r["rank_shift"].items()}
        assert got == want, (r["id"], side)
    score_rows.append({"id": r["id"], "query": r["query"], **t})
print(len(score_rows), "queries; ranks equal the frozen log;", save_or_verify(SCORE_LOG, score_rows))

30 queries; ranks equal the frozen log; verified


## 13. Answers that ran out of output budget

The model's thinking tokens count against the 800-token output budget, so some answers were cut off.
An answer is flagged by a mechanical rule: the saved usage shows `output_tokens` equal to the budget.
For every query with a flagged answer, both of its answers are generated again with a budget of 2000,
so that the pair is compared under one budget. The prompts, the model and the chunks are unchanged.

These are a supplement. The answers in `results/query_log.jsonl` stay the primary record and the metrics
above do not change. The new answers are shown next to the old ones whatever they say, and are kept in
their own cache (`results/answers_cache_cap2000.json`) and log (`results/truncation_supplement.jsonl`).

In [15]:
CAP2 = 2000
SUPP_CACHE = Path("results/answers_cache_cap2000.json")
SUPP_LOG = Path("results/truncation_supplement.jsonl")
cid_index = {c["id"]: i for i, c in enumerate(chunks)}
main_cache = load_cache()

def prompt_of(r, top):
    return build_prompt(r["query"], [cid_index[c] for c in r[top]])

flagged = [r for r in rows for top in ("baseline_top3", "rerank_top3")
           if main_cache[cache_key(prompt_of(r, top))]["usage"]["output_tokens"] >= GEN_MAX_TOKENS]
flagged = list({r["id"]: r for r in flagged}.values())
print("queries with an answer at the budget:", [r["id"] for r in flagged])

if SUPP_LOG.exists():
    supp = load_rows(SUPP_LOG)
else:
    supp = []
    for r in flagged:
        row = {"id": r["id"], "query": r["query"], "budget": CAP2}
        for n, top in ((1, "baseline_top3"), (2, "rerank_top3")):
            user = prompt_of(r, top)
            old = main_cache[cache_key(user)]
            new_answer, new_s, _ = generate(user, CAP2, SUPP_CACHE)
            new = load_cache(SUPP_CACHE)[cache_key(user, CAP2)]
            row[f"answer_{n}_old"] = old["answer"]
            row[f"answer_{n}_new"] = new_answer
            row[f"tokens_{n}_old"] = old["usage"]["output_tokens"]
            row[f"tokens_{n}_new"] = new["usage"]["output_tokens"]
        supp.append(row)
    SUPP_LOG.write_text("".join(json.dumps(r) + "\n" for r in supp))

for r in supp:
    print("=" * 70, "\n", r["id"], r["query"])
    for n in (1, 2):
        print(f"--- answer {n}, old ({r[f'tokens_{n}_old']} tokens):\n{r[f'answer_{n}_old']}")
        print(f"--- answer {n}, new ({r[f'tokens_{n}_new']} tokens):\n{r[f'answer_{n}_new']}")
still = [(r["id"], n) for r in supp for n in (1, 2) if r[f"tokens_{n}_new"] >= CAP2]
print("\nnew answers that still reached the larger budget:", still or "none")

queries with an answer at the budget: ['Q09', 'Q11', 'Q15', 'Q26']
 Q09 How many sets of soft tyres does each driver get at a Standard Format Competition?
--- answer 1, old (429 tokens):
Each driver gets **8 sets of soft-specification tyres** at a Standard Format Competition [1].

The table in [1] is garbled, but its first value column corresponds to the Standard Format Competition. If that competition includes ICTT (In-Competition Tyre Testing), the soft allocation drops to 6 sets [1][3].
--- answer 1, new (492 tokens):
At a Standard Format Competition, each driver is allocated **8 sets of Soft Specification dry-weather tyres** [1].

The table in the passage is garbled, so this reading depends on matching the column order. The first column, labelled Standard Format, gives 8 for Soft. The other columns give 6 (Standard Format with ICTT) and 7 (Alternative Format) [1].
--- answer 2, old (800 tokens):
At a Standard Format Competition, each driver is allocated **8 sets of Soft Specificati

## 14. Exploratory batch: near-duplicate clauses

Written after the frozen run, and a response to it. The 30 frozen queries contain no case where
re-ranking brought a gold chunk into the top 3, so this batch looks at one place where a cross-encoder
could plausibly matter: clauses with almost the same wording that differ in one detail (Sprint and Race,
Sprint Qualifying and Qualifying, Virtual Safety Car and Safety Car, LTCS and TTCS, Standard and
Alternative Format). It is reported as found after seeing the frozen results, it was run once, and it is
never merged into the metrics above.

Rules fixed before the run (this cell and `queries_explore.json` were committed before any of the 14
queries was run through either pipeline):

* The 14 queries were chosen by reading the clause text. No candidate query was run through retrieval
  to see how it behaves.
* The pipelines, the corpus hash, the query prefix, the prompt and the model are the same as above.
* Gold is the rule clause that answers the question, as in the frozen set. A definition that the answer
  also depends on (for example that Qualifying is an LTCS) is named in the note and is not gold. A
  hedged answer caused by a missing definition is therefore not counted as a ranking failure.
* Each query lists a `sibling`: the near-duplicate clause that could be mistaken for the gold clause.
* Restored: the gold chunk is outside the baseline top 3 and inside the re-ranked top 3. Degraded is the
  opposite. Both counts are reported out of 14, with every query listed, misses included.
* Gold above sibling: for each pipeline, how many queries rank the gold chunk above every listed sibling
  (a sibling outside the ten retrieved counts as ranked below). This is the measure of whether a pipeline
  tells near-duplicates apart.
* hit@3 and the reciprocal rank are computed as for the frozen set.
* Prediction, written before the run: the two pipelines will differ little, with few or no restored
  cases, because each query has one clear matching clause. A query where re-ranking changes nothing
  or does harm is reported as such.
* Answers use the same prompt and model. New answers are added to `results/answers_cache.json`. Results go
  to `results/explore_log.jsonl` and `results/explore_score_log.jsonl`, and the cell that runs the batch
  refuses to overwrite either file.


In [16]:
explore = json.loads(Path("queries_explore.json").read_text())
assert all(g in {c["id"] for c in chunks} for q in explore for g in q["gold"] + q["sibling"]), "unknown chunk id"
EXPLORE_LOG = Path("results/explore_log.jsonl")
EXPLORE_SCORES = Path("results/explore_score_log.jsonl")

# Run once: an existing log is read, never rewritten.
explore_rows = load_rows(EXPLORE_LOG) if EXPLORE_LOG.exists() else run_all(explore, log_path=EXPLORE_LOG)
summarise(explore_rows)

explore_scores = []
for r in explore_rows:
    t = score_trace(r["query"])
    for pos, side in ((0, "baseline"), (1, "reranked")):
        got = {c: n for n, (c, _) in enumerate(t[side], 1)}
        assert got == {c: v[pos] for c, v in r["rank_shift"].items()}, (r["id"], side)
    explore_scores.append({"id": r["id"], "query": r["query"], **t})
print(len(explore_scores), "queries; ranks equal the log;", save_or_verify(EXPLORE_SCORES, explore_scores))

type           n | hit@3 base  rerank | recall@3 base  rerank | MRR base  rerank
sprint_vs_race  3 |        1.00    1.00 |           1.00    1.00 |     1.00    1.00
ALL           14 |        0.86    0.93 |           0.86    0.93 |     0.73    0.86
sq_vs_q        2 |        1.00    1.00 |           1.00    1.00 |     0.50    0.75
vsc_vs_sc      4 |        0.75    0.75 |           0.75    0.75 |     0.75    0.62
ltcs_vs_ttcs   2 |        0.50    1.00 |           0.50    1.00 |     0.33    1.00
standard_vs_alternative  3 |        1.00    1.00 |           1.00    1.00 |     0.83    1.00
gold chunk outside the top 10 in both pipelines (re-ranking cannot fix): ['X09']
re-ranking moved the best gold chunk up:   ['X04', 'X10', 'X11', 'X14']
re-ranking moved the best gold chunk down: ['X08']
same top 3 in a different order (answer differences are partly noise): ['X01']


14 queries; ranks equal the log; verified


In [17]:
def explore_table(rows, queries):
    sib = {q["id"]: q["sibling"] for q in queries}
    restored, degraded, above = [], [], {"baseline": 0, "rerank": 0}
    print(f"{'id':<5}{'family':<24}{'gold rank':>12}{'sibling rank':>14}{'gold above sibling':>20}")
    for r in rows:
        g = r["gold"][0]
        base, new = r["rank_shift"].get(g, [None, None])
        line = []
        for k, name in enumerate(("baseline", "rerank")):
            gold_rank = (base, new)[k]
            sib_ranks = [r["rank_shift"][s][k] for s in sib[r["id"]] if s in r["rank_shift"]]
            best_sib = min(sib_ranks) if sib_ranks else TOP_K + 1
            ok = gold_rank is not None and gold_rank < best_sib
            above[name] += ok
            line.append((gold_rank, best_sib if sib_ranks else None, ok))
        if (base is None or base > TOP_N) and new is not None and new <= TOP_N:
            restored.append(r["id"])
        if base is not None and base <= TOP_N and (new is None or new > TOP_N):
            degraded.append(r["id"])
        f = lambda x: "none" if x is None else x
        print(f"{r['id']:<5}{r['type']:<24}{str(f(line[0][0])) + '->' + str(f(line[1][0])):>12}"
              f"{str(f(line[0][1])) + '->' + str(f(line[1][1])):>14}"
              f"{str(line[0][2]) + '->' + str(line[1][2]):>20}")
    n = len(rows)
    print(f"\nrestored (gold outside baseline top 3, inside re-ranked top 3): {len(restored)} of {n}  {restored}")
    print(f"degraded (the opposite): {len(degraded)} of {n}  {degraded}")
    print(f"gold ranked above every listed sibling: baseline {above['baseline']} of {n}, re-ranked {above['rerank']} of {n}")

explore_table(explore_rows, explore)

id   family                     gold rank  sibling rank  gold above sibling
X01  sprint_vs_race                  1->1          3->2          True->True
X02  sprint_vs_race                  1->1          2->2          True->True
X03  sprint_vs_race                  1->1          5->5          True->True
X04  sq_vs_q                         2->1    none->none          True->True
X05  sq_vs_q                         2->2          5->4          True->True
X06  vsc_vs_sc                       1->1    none->none          True->True
X07  vsc_vs_sc                       1->1          2->6          True->True
X08  vsc_vs_sc                       1->2    none->none          True->True
X09  vsc_vs_sc                 none->none    none->none        False->False
X10  ltcs_vs_ttcs                    6->1          4->4         False->True
X11  ltcs_vs_ttcs                    2->1    none->none          True->True
X12  standard_vs_alternative         1->1          2->2          True->True
X13  standar